# FerrisRes: Gemma-4 Distillation on Colab T4

Distill Gemma-4 models into FerrisRes Block AttnRes format.

**Supported models (T4 / 16GB VRAM):**
- **E2B** (~4.6 GB FP16, 35 layers, dense)
- **E4B** (~9.0 GB FP16, 42 layers, dense)

**Too large for T4 (requires A100 80GB):**
- **26B-A4B** (~52 GB FP16, MoE-128)
- **31B** (~62 GB FP16, dense)

The notebook downloads a pre-built binary from the latest GitHub release.
FerrisRes uses profile-driven dispatch — auto-detects the T4 and tiles large matmuls.

**Prerequisites:** Runtime → Change runtime type → **T4 GPU**

In [ ]:
#@title Configuration
model_config = "e2b" #@param ["e2b", "e4b"]
learning_rate = 0.00004 #@param {type:"number"}
seq_len = 256 #@param {type:"integer"}
max_steps = 10000 #@param {type:"integer"}
converge_threshold = 0.001 #@param {type:"number"}
converge_patience = 150 #@param {type:"integer"}

print(f"Config: {model_config}, seq_len={seq_len}, lr={learning_rate}, max_steps={max_steps}")

In [ ]:
# 1. Install Vulkan + Create NVIDIA ICD (Colab has drivers but no Vulkan ICD)
!apt-get update -qq 2>&1 | tail -1
!apt-get install -y -qq libvulkan1 vulkan-tools 2>&1 | tail -3

# Colab has the NVIDIA driver libraries but does NOT ship the Vulkan ICD JSON.
# Without this file, the Vulkan loader can't find the GPU.
import os
icd_dir = "/usr/share/vulkan/icd.d"
nvidia_icd = f"{icd_dir}/nvidia_icd.x86_64.json"
if not os.path.exists(nvidia_icd):
    os.makedirs(icd_dir, exist_ok=True)
    with open(nvidia_icd, 'w') as f:
        f.write('{\n'
                '  "file_format_version": "1.0.0",\n'
                '  "ICD": {\n'
                '    "library_path": "libGLX_nvidia.so.0",\n'
                '    "api_version": "1.3.255"\n'
                '  }\n'
                '}\n')
    print(f"Created {nvidia_icd}")
else:
    print(f"{nvidia_icd} already exists")

# Verify T4 shows in vulkaninfo
!echo '=== Vulkan Devices ===' && vulkaninfo --summary 2>/dev/null | grep deviceName

In [ ]:
# 2. Install FerrisRes
import os, requests

REPO = "shift/FerrisRes"
installed = False

# Try downloading from latest GitHub release
try:
    api = f"https://api.github.com/repos/{REPO}/releases/latest"
    release = requests.get(api, timeout=10).json()
    for asset in release.get('assets', []):
        if asset['name'] == 'ferrisres-x86_64-linux.tar.gz':
            print(f"Downloading from release {release['tag_name']}...")
            r = requests.get(asset['browser_download_url'], timeout=60)
            with open('ferrisres.tar.gz', 'wb') as f:
                f.write(r.content)
            !tar xzf ferrisres.tar.gz && mv ferrisres-x86_64-linux ferrisres && chmod +x ferrisres && rm ferrisres.tar.gz
            print(f"Installed {release['tag_name']}")
            installed = True
            break
    if not installed:
        print(f"No x86_64-linux binary in release {release.get('tag_name', '?')}")
except Exception as e:
    print(f"Release download failed: {e}")

if not installed:
    # Fallback: compile from source (~9 min on T4)
    print("Compiling from source (~9 min on T4)...")
    !curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y 2>&1 | tail -1
    os.environ['PATH'] += ":/root/.cargo/bin"
    !git clone https://github.com/shift/FerrisRes.git /tmp/FerrisRes 2>&1 | tail -1
    !cd /tmp/FerrisRes && cargo build --release --no-default-features --features vulkan 2>&1 | tail -3
    !cp /tmp/FerrisRes/target/release/ferrisres ./ferrisres && chmod +x ferrisres
    print("Compiled from source.")

!./ferrisres --help 2>&1 | head -3

In [ ]:
# 3. Download Model + Tokenizer + Training Data
hf_model_id = f"google/gemma-4-{model_config}-it"
print(f"Downloading {hf_model_id}...")

!wget -q --show-progress https://huggingface.co/{hf_model_id}/resolve/main/model.safetensors
!wget -q https://huggingface.co/{hf_model_id}/resolve/main/tokenizer.json

# WikiText-2 training data (~2MB, ~50K tokens)
!wget -q https://raw.githubusercontent.com/pytorch/examples/master/word_language_model/data/wikitext-2/train.txt -O training_data.txt

!echo '=== Files ===' && ls -lh model.safetensors tokenizer.json training_data.txt

In [ ]:
# 4. Check Hardware Profile
!./ferrisres info

In [ ]:
# 5. Run Distillation
!./ferrisres distill \
  --model-path ./model.safetensors \
  --config {model_config} \
  --seq-len {seq_len} \
  --steps {max_steps} \
  --learning-rate {learning_rate} \
  --temperature 2.0 \
  --tokenizer ./tokenizer.json \
  --data training_data.txt \
  --output distilled_{model_config} \
  --log-every 50 \
  --converge {converge_threshold} \
  --converge-patience {converge_patience}

In [ ]:
# 6. (Optional) Resume if Colab session disconnected
# Uncomment and re-run to continue from last checkpoint:
# !./ferrisres distill \
#   --model-path ./model.safetensors \
#   --config {model_config} \
#   --seq-len {seq_len} \
#   --steps {max_steps} \
#   --learning-rate {learning_rate} \
#   --temperature 2.0 \
#   --tokenizer ./tokenizer.json \
#   --data training_data.txt \
#   --output distilled_{model_config} \
#   --resume distilled_{model_config}.bin.checkpoint \
#   --log-every 50 \
#   --converge {converge_threshold} \
#   --converge-patience {converge_patience}

In [ ]:
# 7. Visualize Distillation Progress
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

loss_file = f'distilled_{model_config}.bin.loss_curve.csv'
try:
    df = pd.read_csv(loss_file)
    df = df.replace('n/a', np.nan)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

    ax1.plot(df['step'], df['kl_loss'], linewidth=2)
    ax1.set_ylabel('KL Divergence Loss')
    ax1.set_title(f'Gemma 4 {model_config.upper()} Distillation')
    ax1.grid(True, linestyle='--', alpha=0.6)

    if 'cosine_sim_avg' in df.columns:
        valid_cos = df.dropna(subset=['cosine_sim_avg'])
        valid_cos['cosine_sim_avg'] = pd.to_numeric(valid_cos['cosine_sim_avg'])
        ax2.plot(valid_cos['step'], valid_cos['cosine_sim_avg'],
                 color='darkorange', marker='o', markersize=3, alpha=0.8)
        ax2.set_ylabel('Cosine Similarity (Teacher vs Student)')
        ax2.set_xlabel('Step')
        ax2.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.show()
except FileNotFoundError:
    print(f'{loss_file} not found. Did the distillation complete?')